In [44]:
import pandas as pd
import os

### 

### Load VMGC metadata

To get list of species to include in this table

In [45]:
vmgc_metadata = pd.read_csv('/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/tables/Table_S1A.csv', index_col=0)
vmgc_metadata

,taxonomy,species_code,representative_genome,num_genomes,mean_percent_completeness,std_percent_completeness,range_percent_completeness,mean_percent_contamination,std_percent_contamination,range_percent_contamination,...,range_quality_score,mean_num_bp_per_genome,std_num_bp_per_genome,range_num_bp_per_genome,num_genes_in_C99_pangenome,num_genes_in_C95_pangenome,num_genes_in_C90_pangenome,num_genes_in_C85_pangenome,num_genes_in_C80_pangenome,num_genes_in_C75_pangenome
species_name,,,,,,,,,,,,,,,,,,,,,
Acinetobacter baumannii,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,619896,GCF_000162295.1,5,90.136000,22.034221,"(50.72, 100.0)",0.140000,0.074162,"(0.01, 0.19)",...,"(50.67, 99.25)",3.503209e+06,1.028650e+06,"(1672604, 4045681)",7549,4939,4691,4635,4592,4559
Actinobaculum massiliense,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,629828,GCF_000315465.1,5,80.300000,18.247766,"(58.14, 98.49)",1.302000,0.944442,"(0.16, 2.75)",...,"(53.09, 93.67)",1.659384e+06,3.788633e+05,"(1272294, 2071203)",1899,1889,1886,1884,1880,1875
Actinotignum sanguinis,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,858790,SRR17635715.sbin.92,5,86.468000,11.984729,"(72.59, 98.85)",0.884000,1.180436,"(0.09, 2.9)",...,"(62.83, 98.4)",1.842602e+06,2.571240e+05,"(1525935, 2154979)",2643,2282,2112,2043,2004,1984
Aerococcus christensenii,d__Bacteria;p__Bacillota;c__Bacilli;o__Lactoba...,542621,P10709641.mbin.2,260,85.814500,11.683608,"(52.76, 99.97)",0.935269,0.840510,"(0.04, 4.74)",...,"(50.17, 99.03)",1.295191e+06,1.911237e+05,"(770800, 1710611)",8274,3451,2933,2738,2642,2555
Alistipes putredinis,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,712851,SAMN00040503.mbin.1,5,89.836000,11.093319,"(76.34, 99.99)",1.464000,1.811555,"(0.41, 4.68)",...,"(52.94, 97.89)",2.021397e+06,2.853255e+05,"(1659934, 2321816)",4064,2944,2838,2806,2781,2758
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Varibaculum massiliense,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,904423,ERR10897914.mbin.4,12,87.870000,8.595570,"(77.09, 99.99)",1.257500,0.735083,"(0.23, 2.94)",...,"(66.37, 98.57)",1.776246e+06,2.250029e+05,"(1457354, 2101645)",9558,4197,2990,2643,2550,2497
Veillonella montpellierensis,d__Bacteria;p__Bacillota_C;c__Negativicutes;o_...,674615,SRR16470998.sbin.6,39,74.209744,15.710640,"(52.36, 99.99)",0.552821,0.669233,"(0.0, 2.18)",...,"(51.41, 97.04)",1.333022e+06,3.377077e+05,"(781301, 1976838)",12344,3756,3149,2979,2902,2853
Veillonella parvula_A,d__Bacteria;p__Bacillota_C;c__Negativicutes;o_...,773494,GCF_000215025.2,6,93.465000,8.993808,"(78.14, 100.0)",0.803333,1.529898,"(0.08, 3.92)",...,"(58.54, 99.44)",1.902467e+06,2.154803e+05,"(1574768, 2152140)",6875,3504,2660,2468,2410,2374


### Load GTDB MIDAS metadata

In [46]:
df = pd.read_csv('MIDAS_GTDB_metadata_by_species.tsv', sep='\t')
df = df.rename(columns={'No. clustered genomes': 'num_genomes',
                        'GTDB taxonomy':'taxonomy',
                        'species_id':'species_code',
                        'GTDB species':'species_name'
                        })
df = df[['species_name', 'taxonomy', 'species_code', 'representative', 'num_genomes' ]]
df['species_name'] = df['species_name'].str.replace('s__','')
df = df[df['species_name'].isin(vmgc_metadata.index)].set_index('species_name')
df.shape


(128, 4)

In [47]:
df.head()


,taxonomy,species_code,representative,num_genomes
species_name,,,,
Staphylococcus aureus,d__Bacteria;p__Firmicutes;c__Bacilli;o__Staphy...,100002,GCF_001027105.1,12022
Klebsiella pneumoniae,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,100004,GCF_000742135.1,9621
Acinetobacter baumannii,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,100008,GCF_009759685.1,4766
Escherichia coli,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,100009,GCF_003697165.2,4370
Streptococcus pyogenes,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,100010,GCF_002055535.1,2136


### Get quality info from NCBI


In [48]:
genome_info = pd.read_csv('GTDB_genomes_in_MIDAS.tsv', sep='\t')
genome_info = genome_info[genome_info['species'].isin(df['species_code'].values)]
genome_info[['genome']].to_csv('accession_list.txt', header=False, index=False)


Running this in command line:
        
        datasets summary genome accession --inputfile accession_list.txt --as-json-lines |  
        dataformat tsv genome --fields accession,checkm-contamination,checkm-completeness,assmstats-total-sequence-len --force > NCBI_genome_quality_info.tsv



In [49]:
quality_info = pd.read_csv('NCBI_genome_quality_info.tsv', sep='\t', index_col=0)
quality_info = quality_info.rename(columns={'CheckM contamination': '%_Contamination',
                                            'CheckM completeness':  '%_Completeness',
                                            'Assembly Stats Total Sequence Length': 'Genome_size_(bp)'})

#Calculate quality score like they did in VMGC paper: %completeness − 5×%contamination
quality_info['%_Completeness'] = quality_info['%_Completeness'].round(2)
quality_info['%_Contamination'] = quality_info['%_Contamination'].round(2)
quality_info['Quality_score'] = quality_info['%_Completeness'] - (5*quality_info['%_Contamination'])


quality_info = quality_info.join(genome_info.set_index('genome').drop(columns=['representative']).rename(columns={'species':'species_code'})).reset_index()
merged = df.reset_index().merge(quality_info, left_on='species_code', right_on='species_code', how='right')

In [50]:
merged.head()

,species_name,taxonomy,species_code,representative,num_genomes,Assembly Accession,%_Contamination,%_Completeness,Genome_size_(bp),Quality_score,genome_is_representative
0,Staphylococcus aureus,d__Bacteria;p__Firmicutes;c__Bacilli;o__Staphy...,100002,GCF_001027105.1,12022,GCF_000418345.1,0.47,98.37,2983456,96.02,0
1,Staphylococcus aureus,d__Bacteria;p__Firmicutes;c__Bacilli;o__Staphy...,100002,GCF_001027105.1,12022,GCA_000647755.1,1.64,92.70,2819947,84.50,0
2,Staphylococcus aureus,d__Bacteria;p__Firmicutes;c__Bacilli;o__Staphy...,100002,GCF_001027105.1,12022,GCA_000709475.1,0.19,93.66,3052055,92.71,0
3,Staphylococcus aureus,d__Bacteria;p__Firmicutes;c__Bacilli;o__Staphy...,100002,GCF_001027105.1,12022,GCA_000732745.1,0.67,77.83,2911065,74.48,0
4,Staphylococcus aureus,d__Bacteria;p__Firmicutes;c__Bacilli;o__Staphy...,100002,GCF_001027105.1,12022,GCA_000934185.1,0.39,98.11,2756431,96.16,0


### Group by species

In [51]:
# helper function to calculate the range and round the results
get_rounded_range = lambda x: (round(x.min(), 2), round(x.max(), 2))

species_df = merged.groupby('species_name').agg(
    species_code=('species_code', 'first'),
    representative_genome=('representative', 'first'),
    taxonomy=('taxonomy', 'first'),
    num_genomes=('Assembly Accession', 'count'),

    mean_quality_score=('Quality_score', 'mean'),
    std_quality_score=('Quality_score', 'std'),
    range_quality_score=('Quality_score', get_rounded_range),

    mean_percent_completeness=('%_Completeness', 'mean'),
    std_percent_completeness=('%_Completeness', 'std'),
    range_percent_completeness=('%_Completeness', get_rounded_range),

    mean_percent_contamination=('%_Contamination', 'mean'),
    std_percent_contamination=('%_Contamination', 'std'),
    range_percent_contamination=('%_Contamination', get_rounded_range),

    mean_num_bp_per_genome=('Genome_size_(bp)', 'mean'),
    std_num_bp_per_genome=('Genome_size_(bp)', 'std'),
    range_num_bp_per_genome=('Genome_size_(bp)', get_rounded_range),
)

species_df = species_df.round(2)
species_df


,species_code,representative_genome,taxonomy,num_genomes,mean_quality_score,std_quality_score,range_quality_score,mean_percent_completeness,std_percent_completeness,range_percent_completeness,mean_percent_contamination,std_percent_contamination,range_percent_contamination,mean_num_bp_per_genome,std_num_bp_per_genome,range_num_bp_per_genome
species_name,,,,,,,,,,,,,,,,
Acinetobacter baumannii,100008,GCF_009759685.1,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,4766,95.98,5.28,"(27.51, 99.84)",99.22,2.02,"(47.57, 99.99)",0.65,0.95,"(0.02, 14.12)",3980088.06,130110.89,"(2951520, 4777150)"
Actinobaculum massiliense,111310,GCF_001457435.1,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,2,96.85,0.00,"(96.85, 96.85)",99.10,0.00,"(99.1, 99.1)",0.45,0.00,"(0.45, 0.45)",2048600.50,23454.02,"(2032016, 2065185)"
Actinotignum sanguinis,146291,GCF_003957255.1,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,1,NaN,NaN,"(nan, nan)",98.66,NaN,"(98.66, 98.66)",NaN,NaN,"(nan, nan)",2038007.00,NaN,"(2038007, 2038007)"
Aerococcus christensenii,105820,GCF_001543105.1,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,4,58.42,3.53,"(53.13, 60.27)",75.13,0.06,"(75.09, 75.22)",3.34,0.70,"(2.99, 4.4)",1649849.00,50944.12,"(1589378, 1710611)"
Alistipes putredinis,101896,GCF_000154465.1,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,9,84.73,11.18,"(65.35, 97.26)",91.24,9.16,"(75.55, 98.76)",1.30,0.89,"(0.3, 2.86)",2150194.00,324620.25,"(1721247, 2550678)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Urinicoccus massiliensis,109785,GCF_900120405.1,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__T...,2,NaN,NaN,"(nan, nan)",97.42,1.22,"(96.55, 98.28)",NaN,NaN,"(nan, nan)",2082859.00,6082.53,"(2078558, 2087160)"
Varibaculum cambriense_B,107685,GCF_002848005.1,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,3,90.12,8.70,"(83.97, 96.27)",97.67,1.63,"(96.52, 98.82)",1.51,1.41,"(0.51, 2.51)",2158292.33,136520.45,"(2035249, 2305155)"
Varibaculum massiliense,124449,GCF_900106855.1,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,1,96.10,NaN,"(96.1, 96.1)",99.05,NaN,"(99.05, 99.05)",0.59,NaN,"(0.59, 0.59)",2283357.00,NaN,"(2283357, 2283357)"


### Get pangenome sizes

In [52]:
pangenome_counts = []
cluster_levels = [75, 80, 85, 90, 95, 99]
gtdb_species_to_id = species_df['species_code'].to_dict()

for sp in species_df.index:

    v = gtdb_species_to_id[sp]
    cluster_path = f'GTDB_db/pangenomes/{v}/clusters_99_info.tsv'
    if not os.path.exists(cluster_path):
        print(v)
        continue

    gtdb_clusters = pd.read_csv(cluster_path, sep='\t')   

    for c in cluster_levels:
        
        gtdb_count = gtdb_clusters[f'centroid_{c}'].unique().shape[0]
        
        pangenome_counts += [[sp, v, c, gtdb_count,]]


/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_99877/1616661083.py:13: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  gtdb_clusters = pd.read_csv(cluster_path, sep='\t')


In [53]:
pangenome_counts = pd.DataFrame(pangenome_counts)
pangenome_counts.columns = ['species_name', 'GTDB_species_code', 'cluster_level', 'GTDB_pangenome_size']
pangenome_counts['num_genomes'] = pangenome_counts['species_name'].map(species_df['num_genomes'])
# pangenome_counts.to_csv('GTDB_pangenome_counts.csv', index=False)

In [54]:
pangenome_counts_piv = pangenome_counts.pivot(index='species_name', columns='cluster_level', values='GTDB_pangenome_size')
pangenome_counts_piv.columns = [f'num_genes_in_C{i}_pangenome' for i in pangenome_counts_piv.columns]

In [55]:
pangenome_counts_piv = pangenome_counts_piv[pangenome_counts_piv.columns[::-1]]
pangenome_counts_piv.head()

,num_genes_in_C99_pangenome,num_genes_in_C95_pangenome,num_genes_in_C90_pangenome,num_genes_in_C85_pangenome,num_genes_in_C80_pangenome,num_genes_in_C75_pangenome
species_name,,,,,,
Acinetobacter baumannii,208927,63965,41284,33554,30531,29134
Actinobaculum massiliense,1708,1703,1700,1700,1698,1695
Actinotignum sanguinis,1641,1640,1640,1639,1633,1630
Aerococcus christensenii,2193,1766,1732,1721,1701,1685
Alistipes putredinis,5243,3536,3343,3291,3255,3222


In [56]:
species_df = species_df.join(pangenome_counts_piv)

In [57]:
#reorder columns so they are the same as the VMGC table
species_df = species_df[['taxonomy', 'species_code', 'representative_genome', 'num_genomes',
      
       'mean_percent_completeness', 'std_percent_completeness',
       'range_percent_completeness', 'mean_percent_contamination',
       'std_percent_contamination', 'range_percent_contamination',
        'mean_quality_score', 'std_quality_score', 'range_quality_score',
       'mean_num_bp_per_genome', 'std_num_bp_per_genome',
       'range_num_bp_per_genome', 'num_genes_in_C99_pangenome',
       'num_genes_in_C95_pangenome', 'num_genes_in_C90_pangenome',
       'num_genes_in_C85_pangenome', 'num_genes_in_C80_pangenome',
       'num_genes_in_C75_pangenome']]

In [58]:
species_df

,taxonomy,species_code,representative_genome,num_genomes,mean_percent_completeness,std_percent_completeness,range_percent_completeness,mean_percent_contamination,std_percent_contamination,range_percent_contamination,...,range_quality_score,mean_num_bp_per_genome,std_num_bp_per_genome,range_num_bp_per_genome,num_genes_in_C99_pangenome,num_genes_in_C95_pangenome,num_genes_in_C90_pangenome,num_genes_in_C85_pangenome,num_genes_in_C80_pangenome,num_genes_in_C75_pangenome
species_name,,,,,,,,,,,,,,,,,,,,,
Acinetobacter baumannii,d__Bacteria;p__Proteobacteria;c__Gammaproteoba...,100008,GCF_009759685.1,4766,99.22,2.02,"(47.57, 99.99)",0.65,0.95,"(0.02, 14.12)",...,"(27.51, 99.84)",3980088.06,130110.89,"(2951520, 4777150)",208927,63965,41284,33554,30531,29134
Actinobaculum massiliense,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,111310,GCF_001457435.1,2,99.10,0.00,"(99.1, 99.1)",0.45,0.00,"(0.45, 0.45)",...,"(96.85, 96.85)",2048600.50,23454.02,"(2032016, 2065185)",1708,1703,1700,1700,1698,1695
Actinotignum sanguinis,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,146291,GCF_003957255.1,1,98.66,NaN,"(98.66, 98.66)",NaN,NaN,"(nan, nan)",...,"(nan, nan)",2038007.00,NaN,"(2038007, 2038007)",1641,1640,1640,1639,1633,1630
Aerococcus christensenii,d__Bacteria;p__Firmicutes;c__Bacilli;o__Lactob...,105820,GCF_001543105.1,4,75.13,0.06,"(75.09, 75.22)",3.34,0.70,"(2.99, 4.4)",...,"(53.13, 60.27)",1649849.00,50944.12,"(1589378, 1710611)",2193,1766,1732,1721,1701,1685
Alistipes putredinis,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,101896,GCF_000154465.1,9,91.24,9.16,"(75.55, 98.76)",1.30,0.89,"(0.3, 2.86)",...,"(65.35, 97.26)",2150194.00,324620.25,"(1721247, 2550678)",5243,3536,3343,3291,3255,3222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Urinicoccus massiliensis,d__Bacteria;p__Firmicutes_A;c__Clostridia;o__T...,109785,GCF_900120405.1,2,97.42,1.22,"(96.55, 98.28)",NaN,NaN,"(nan, nan)",...,"(nan, nan)",2082859.00,6082.53,"(2078558, 2087160)",3253,2240,2145,2114,2095,2083
Varibaculum cambriense_B,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,107685,GCF_002848005.1,3,97.67,1.63,"(96.52, 98.82)",1.51,1.41,"(0.51, 2.51)",...,"(83.97, 96.27)",2158292.33,136520.45,"(2035249, 2305155)",3957,2923,2474,2353,2299,2266
Varibaculum massiliense,d__Bacteria;p__Actinobacteriota;c__Actinomycet...,124449,GCF_900106855.1,1,99.05,NaN,"(99.05, 99.05)",0.59,NaN,"(0.59, 0.59)",...,"(96.1, 96.1)",2283357.00,NaN,"(2283357, 2283357)",1916,1910,1900,1895,1888,1880


In [59]:
species_df.to_csv('/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/tables/Table_S1B.csv')

### Looking for overlap between GTDB and VMGC genomes

In [60]:
gtdb_genomes = pd.read_csv('GTDB_genomes_in_MIDAS.tsv', sep='\t')
gtdb_genomes['genome_no_version'] = gtdb_genomes['genome'].str.split('.', expand=True)[0]
gtdb_genomes['genome_no_version'] = gtdb_genomes['genome_no_version'].str.replace(r'^GCA', 'GCF', regex=True)
gtdb_genomes.head()

,genome,species,representative,genome_is_representative,genome_no_version
0,GCA_000092525.1,100001,GCF_002950215.1,0,GCF_000092525
1,GCA_000165655.2,100001,GCF_002950215.1,0,GCF_000165655
2,GCA_000188795.2,100001,GCF_002950215.1,0,GCF_000188795
3,GCA_000188815.2,100001,GCF_002950215.1,0,GCF_000188815
4,GCA_000193935.2,100001,GCF_002950215.1,0,GCF_000193935


In [61]:
vmgc_genomes = pd.read_csv('/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/code/VMGC/VMGC_db_build/genomes.tsv', sep='\t')

vmgc_genomes['genome_no_version'] = vmgc_genomes['genome'].apply(
    lambda x: x.split('.')[0] if str(x).startswith(('GCF', 'GCA')) else x
)
vmgc_genomes['genome_no_version'] = vmgc_genomes['genome_no_version'].str.replace(r'^GCA', 'GCF', regex=True)
vmgc_genomes.head()

,genome,species,representative,genome_is_representative,fasta_path,genome_no_version
0,GCF_001722255.1,235527,GCF_001722255.1,1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,GCF_001722255
1,GCF_001679135.1,945663,GCF_001679135.1,1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,GCF_001679135
2,GCF_001811805.1,235527,GCF_001722255.1,0,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,GCF_001811805
3,GCF_009495705.1,429734,GCF_009495705.1,1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,GCF_009495705
4,GCF_000439915.2,611554,GCF_000439915.2,1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...,GCF_000439915


In [62]:
vmgc_genomes['genome_no_version'].str.startswith('GCF').sum()

np.int64(972)

In [63]:
gtdb_genome_list = gtdb_genomes['genome_no_version'].values
shared = [i for i in vmgc_genomes['genome_no_version'] if i in gtdb_genome_list]
len(shared)

624

In [68]:
shared = [i for i in vmgc_genomes[vmgc_genomes['species'].isin([611554, 240891, 988598, 571325, 619501, 783244])]['genome_no_version'] if i in gtdb_genome_list]
len(shared)

100

In [70]:
a = ['GCF_000439915.2',
 'GCF_000175055.1',
 'GCF_011058715.1',
 'GCF_000176995.2',
 'GCF_000179955.1',
 'GCF_000179995.1',
 'GCF_000162255.1',
 'GCF_011058795.1',
 'GCF_011058695.1',
 'GCF_011058775.1',
 'GCF_011058755.1',
 'GCF_009857205.1',
 'GCF_000162335.1',
 'GCF_000160875.1',
 'GCF_000177035.2',
 'GCF_008868575.1',
 'GCF_000175035.1',
 'GCF_011029225.1',
 'GCF_001936235.1',
 'GCF_011058735.1',
 'GCF_004681235.1',
 'GCF_000185405.1',
 'GCF_001541385.1',
 'GCF_008079315.1',
 'GCF_000177575.1',
 'GCF_000176975.2',
 'GCF_004361575.1',
 'GCF_000160515.1',
 'GCF_014654855.1',
 'GCF_004361375.1',
 'GCF_014654865.1',
 'GCF_001562845.1',
 'GCF_004361295.1',
 'GCF_000301115.1',
 'GCF_004361265.1',
 'GCF_004361635.1',
 'GCF_000466885.3',
 'GCF_004361465.1',
 'GCF_000159235.2',
 'GCF_900445305.1',
 'GCF_001541535.1',
 'GCF_004361475.1',
 'GCF_000161915.2',
 'GCF_001546015.1',
 'GCF_004361355.1',
 'GCF_004361565.1',
 'GCF_011029265.1',
 'GCF_001541405.1',
 'GCF_001546025.1',
 'GCF_009730275.1',
 'GCF_003971565.1',
 'GCF_004103355.1',
 'GCF_004361555.1',
 'GCF_004361515.1',
 'GCF_000179935.1',
 'GCF_004361395.1',
 'GCF_004361455.1',
 'GCF_001541515.1',
 'GCF_004361185.1',
 'GCF_002892385.1',
 'GCF_000301135.1',
 'GCF_000162315.1',
 'GCF_001541585.1',
 'GCF_004361385.1',
 'GCF_003795065.1',
 'GCF_000155935.2',
 'GCF_004361545.1',
 'GCF_004361345.1',
 'GCF_000149065.1',
 'GCF_000177415.1',
 'GCF_001541505.1',
 'GCF_000149085.1',
 'GCF_000155915.2',
 'GCF_001049775.1',
 'GCF_010748955.1',
 'GCF_000149105.1',
 'GCF_001436455.1',
 'GCF_000179975.1',
 'GCF_003397605.1',
 'GCF_000213955.1',
 'GCF_001049785.1',
 'GCF_002894105.1',
 'GCF_900637625.1',
 'GCF_003397685.1',
 'GCF_001042655.1',
 'GCF_000149145.1',
 'GCF_003408785.1',
 'GCF_003408745.1',
 'GCF_000159155.2',
 'GCF_003397665.1',
 'GCF_003585755.1',
 'GCF_000414645.1',
 'GCF_000149125.1',
 'GCF_000414705.1',
 'GCF_003585655.1',
 'GCF_000165885.1']

[i for i in shared if i not in [b.split('.')[0] for b in a ]]

['GCF_000191685', 'GCF_000191705', 'GCF_000204435', 'GCF_000214315']

In [64]:
972/(972+18570)

0.04973902364138778

In [65]:
len(shared)/vmgc_genomes.shape[0]

0.03193122505373042

In [40]:
vmgc_genomes.shape

(19542, 6)

In [21]:
gtdb_genomes

,genome,species,representative,genome_is_representative
0,GCA_000092525.1,100001,GCF_002950215.1,0
1,GCA_000165655.2,100001,GCF_002950215.1,0
2,GCA_000188795.2,100001,GCF_002950215.1,0
3,GCA_000188815.2,100001,GCF_002950215.1,0
4,GCA_000193935.2,100001,GCF_002950215.1,0
...,...,...,...,...
258400,GCA_002315235.1,147890,GCA_002315235.1,1
258401,GCA_002498315.1,147891,GCA_002498315.1,1
258402,GCA_000522425.1,147892,GCA_000522425.1,1
258403,GCA_009780065.1,147893,GCA_009780065.1,1


In [23]:
gtdb_genomes[gtdb_genomes['genome'].str.startswith('GC')]

,genome,species,representative,genome_is_representative
0,GCA_000092525.1,100001,GCF_002950215.1,0
1,GCA_000165655.2,100001,GCF_002950215.1,0
2,GCA_000188795.2,100001,GCF_002950215.1,0
3,GCA_000188815.2,100001,GCF_002950215.1,0
4,GCA_000193935.2,100001,GCF_002950215.1,0
...,...,...,...,...
258400,GCA_002315235.1,147890,GCA_002315235.1,1
258401,GCA_002498315.1,147891,GCA_002498315.1,1
258402,GCA_000522425.1,147892,GCA_000522425.1,1
258403,GCA_009780065.1,147893,GCA_009780065.1,1


In [24]:
gtdb_genomes[gtdb_genomes['genome'].str.startswith('SRR')]

,genome,species,representative,genome_is_representative
